In [1]:
from datasets import load_dataset

## HelperSteer2-Disagreement - Data Transformation

In [2]:
disagreements = load_dataset("nvidia/HelpSteer2", data_dir="disagreements")['train']
disagreements[2]

{'prompt': 'bacillus subtilus',
 'response': 'Bacillus subtilis is a Gram-positive, rod-shaped bacterium commonly found in soil and water. It is a facultative anaerobe that can grow in the presence or absence of oxygen. B. subtilis is a model organism for studying bacterial cell biology, genetics, and physiology, and has many industrial and medical applications.\n\nB. subtilis is a versatile organism with many interesting properties. It can form spores that allow it to survive harsh environmental conditions, such as heat, cold, and dehydration. The spores can remain dormant for long periods of time and then germinate when conditions are favorable. B. subtilis can also produce a variety of enzymes, antibiotics, and other compounds that have industrial and medical applications.\n\nB. subtilis has been used in the production of enzymes for industrial applications, such as in the food and beverage industry for the production of cheese, yogurt, and beer. It is also used in the production of

In [3]:
c_sharp_examples = [ex for ex in disagreements if ex['prompt'] == 'c#']
c_sharp_examples

[{'prompt': 'c#',
  'response': 'C# is a high-level, object-oriented programming language developed by Microsoft as part of its .NET initiative. It was created as a modern alternative to Java and supports a variety of programming paradigms, including imperative, functional, and event-driven. C# is primarily used for Windows application development, but it can also be used for web, mobile, and game development. The language is designed to be safe, secure, and efficient, and it provides developers with a rich set of libraries and tools for building robust and scalable applications. C# is also widely used in the game development industry, particularly in the development of games for the Xbox 360 and Xbox One consoles.',
  'helpfulness': [3, 3, 4],
  'correctness': [3, 4, 4],
  'coherence': [4, 3, 4],
  'complexity': [2, 2, 2],
  'verbosity': [2, 1, 1]},
 {'prompt': 'c#',
  'response': 'C# (pronounced "C sharp") is a modern, object-oriented programming language developed by Microsoft. It i

In [4]:
import pandas as pd
from collections import defaultdict

def transform_disagreement_data_paired(dataset):
    """
    Transform disagreement dataset where responses are paired by prompt.
    Each row represents one annotator's evaluation of both responses.
    
    Format: prompt, response1, response2, annotatorID, 
            helpfulness1, correctness1, coherence1, complexity1, verbosity1,
            helpfulness2, correctness2, coherence2, complexity2, verbosity2
    """
    prompt_groups = defaultdict(list)
    for example in dataset:
        prompt_groups[example['prompt']].append(example)
    
    transformed_data = []
    
    for prompt, responses in prompt_groups.items():
        if len(responses) == 2:
            response1 = responses[0]
            response2 = responses[1]
            
            num_annotators = min(len(response1['helpfulness']), len(response2['helpfulness']))
            
            for annotator_id in range(num_annotators):
                row = {
                    'prompt': prompt,
                    'response1': response1['response'],
                    'response2': response2['response'],
                    'annotatorID': annotator_id,
                    'helpfulness1': response1['helpfulness'][annotator_id],
                    'correctness1': response1['correctness'][annotator_id],
                    'coherence1': response1['coherence'][annotator_id],
                    'complexity1': response1['complexity'][annotator_id],
                    'verbosity1': response1['verbosity'][annotator_id],
                    'helpfulness2': response2['helpfulness'][annotator_id],
                    'correctness2': response2['correctness'][annotator_id],
                    'coherence2': response2['coherence'][annotator_id],
                    'complexity2': response2['complexity'][annotator_id],
                    'verbosity2': response2['verbosity'][annotator_id],
                }
                transformed_data.append(row)
        else:
            print(f"Warning: Prompt '{prompt[:50]}...' has {len(responses)} response(s), expected 2. Skipping.")
    
    return pd.DataFrame(transformed_data)

df_transformed = transform_disagreement_data_paired(disagreements)
print(f"Original dataset size: {len(disagreements)}")
print(f"Number of unique prompts: {len(set(ex['prompt'] for ex in disagreements))}")
print(f"Transformed dataset size: {len(df_transformed)}")
print(f"\nFirst few rows:")
df_transformed.head(10)

Original dataset size: 23652
Number of unique prompts: 11825
Transformed dataset size: 40365

First few rows:


,prompt,response1,response2,annotatorID,helpfulness1,correctness1,coherence1,complexity1,verbosity1,helpfulness2,correctness2,coherence2,complexity2,verbosity2
0,c#,"C# is a high-level, object-oriented programmin...","C# (pronounced ""C sharp"") is a modern, object-...",0,3,3,4,2,2,4,4,4,2,4
1,c#,"C# is a high-level, object-oriented programmin...","C# (pronounced ""C sharp"") is a modern, object-...",1,3,4,3,2,1,4,4,4,2,2
2,c#,"C# is a high-level, object-oriented programmin...","C# (pronounced ""C sharp"") is a modern, object-...",2,4,4,4,2,1,4,4,4,3,3
3,bacillus subtilus,"Bacillus subtilis is a Gram-positive, rod-shap...","The Bacillus subtilis is a Gram-positive, endo...",0,4,4,4,3,2,4,4,4,3,1
4,bacillus subtilus,"Bacillus subtilis is a Gram-positive, rod-shap...","The Bacillus subtilis is a Gram-positive, endo...",1,4,4,4,4,3,3,3,3,4,2
5,bacillus subtilus,"Bacillus subtilis is a Gram-positive, rod-shap...","The Bacillus subtilis is a Gram-positive, endo...",2,3,4,3,2,3,3,4,3,2,3
6,Write long detailed essay about ancient type o...,Totemism is an ancient type of religion that i...,Totemism is an ancient religious and philosoph...,0,2,2,4,1,2,4,4,4,1,3
7,Write long detailed essay about ancient type o...,Totemism is an ancient type of religion that i...,Totemism is an ancient religious and philosoph...,1,3,4,3,2,3,3,4,3,2,3
8,Write long detailed essay about ancient type o...,Totemism is an ancient type of religion that i...,Totemism is an ancient religious and philosoph...,2,3,3,3,2,2,2,2,2,2,2
9,Define Signal Discuss its various properties w...,A signal is a form of energy that is used to t...,A signal is a message that is conveyed from a ...,0,3,3,3,1,2,3,4,3,1,2


In [5]:
# Save the transformed dataset
output_path = "train/helpsteer2_disagreement_paired.json"
df_transformed.to_json(output_path, orient='records', indent=2)
print(f"Transformed dataset saved to: {output_path}")

Transformed dataset saved to: train/helpsteer2_disagreement_paired.json


## MultiPref-Disagreements

In [6]:
multipref = load_dataset("allenai/multipref", data_dir="data")['train']

In [7]:
def transform_multipref_data(dataset, use_expert_annotations=False):
    """
    Transform MultiPref dataset to have one row per annotator per comparison.
    
    Format: prompt, response1, response2, annotatorID, comparison_id,
            helpful, truthful, harmless, overall,
            helpful_confidence, truthful_confidence, harmless_confidence, overall_confidence
    
    Preference encoding:
    - A-is-clearly-better: 2
    - A-is-slightly-better: 1
    - Tie: 0
    - B-is-slightly-better: -1
    - B-is-clearly-better: -2
    """

    pref_encoding = {
        'A-is-clearly-better': 2,
        'A-is-slightly-better': 1,
        'Tie': 0,
        'B-is-slightly-better': -1,
        'B-is-clearly-better': -2
    }
    
    transformed_data = []
    
    for example in dataset:
        annotations = example['expert_worker_annotations'] if use_expert_annotations else example['normal_worker_annotations']
        
        if not annotations:
            continue
        
        for annot in annotations:
            row = {
                'comparison_id': example['comparison_id'],
                'prompt': example['text'],
                'response1': example['completion_a'],
                'response2': example['completion_b'],
                'annotatorID': annot['evaluator'],
                'helpful': pref_encoding.get(annot['helpful_pref'], 0),
                'truthful': pref_encoding.get(annot['truthful_pref'], 0),
                'harmless': pref_encoding.get(annot['harmless_pref'], 0),
                'overall': pref_encoding.get(annot['overall_pref'], 0),
                'helpful_confidence': annot['helpful_confidence'],
                'truthful_confidence': annot['truthful_confidence'],
                'harmless_confidence': annot['harmless_confidence'],
                'overall_confidence': annot['overall_confidence'],
            }
            transformed_data.append(row)
    
    return pd.DataFrame(transformed_data)

In [8]:
import pandas as pd
from collections import defaultdict

In [9]:
# Apply transformation for normal worker annotations
df_multipref_normal = transform_multipref_data(multipref, use_expert_annotations=False)

print(f"\n\nFirst few rows of transformed data:")
df_multipref_normal.head(10)



First few rows of transformed data:


,comparison_id,prompt,response1,response2,annotatorID,helpful,truthful,harmless,overall,helpful_confidence,truthful_confidence,harmless_confidence,overall_confidence
0,cdccd42baceb4358888fda8c23a7fb32,but it will provide acceleration in y directio...,To calculate the distance traveled in the y-di...,To find the distance traveled in the y-directi...,mystifying_fermi,2,2,0,2,absolutely-confident,fairly-confident,absolutely-confident,absolutely-confident
1,cdccd42baceb4358888fda8c23a7fb32,but it will provide acceleration in y directio...,To calculate the distance traveled in the y-di...,To find the distance traveled in the y-directi...,unruffled_hoover,2,0,0,2,absolutely-confident,absolutely-confident,absolutely-confident,absolutely-confident
2,aa8af1e5ea024ad08e71273f5695993b,what are some fun sandwiches i can make for kids?,Here are a few fun sandwich ideas that kids mi...,"Kids love sandwiches, and making them fun can ...",romantic_liskov,-1,0,0,-1,fairly-confident,absolutely-confident,absolutely-confident,fairly-confident
3,aa8af1e5ea024ad08e71273f5695993b,what are some fun sandwiches i can make for kids?,Here are a few fun sandwich ideas that kids mi...,"Kids love sandwiches, and making them fun can ...",determined_elion,-2,0,0,0,fairly-confident,fairly-confident,fairly-confident,fairly-confident
4,6b9671fb535b447e898a24771d136306,what are some fun sandwiches i can make for kids?,Here are a few ideas for fun sandwiches that k...,Making sandwiches for kids is not only about p...,practical_agnesi,-2,0,0,-2,absolutely-confident,absolutely-confident,absolutely-confident,absolutely-confident
5,6b9671fb535b447e898a24771d136306,what are some fun sandwiches i can make for kids?,Here are a few ideas for fun sandwiches that k...,Making sandwiches for kids is not only about p...,elastic_swanson,-2,0,0,-2,absolutely-confident,absolutely-confident,absolutely-confident,absolutely-confident
6,924dbf961d664b1c8634a0d831f0c8d4,what are august activiites for copenhagen rank...,"I'm sorry, but I don't have information about ...",Here are some popular activities to do in Co...,suspicious_saha,-2,-2,0,-2,absolutely-confident,absolutely-confident,absolutely-confident,absolutely-confident
7,924dbf961d664b1c8634a0d831f0c8d4,what are august activiites for copenhagen rank...,"I'm sorry, but I don't have information about ...",Here are some popular activities to do in Co...,pedantic_kowalevski,-2,-1,0,-2,absolutely-confident,absolutely-confident,absolutely-confident,absolutely-confident
8,fb0315fba0f1484994e2a9b3e7e1d4cb,what are august activiites for copenhagen rank...,Here are some popular activities to do in Cope...,"I'm sorry, but I don't have information about ...",unruffled_hoover,2,0,0,2,absolutely-confident,absolutely-confident,absolutely-confident,absolutely-confident
9,fb0315fba0f1484994e2a9b3e7e1d4cb,what are august activiites for copenhagen rank...,Here are some popular activities to do in Cope...,"I'm sorry, but I don't have information about ...",vigorous_lovelace,-2,-2,0,-2,absolutely-confident,absolutely-confident,absolutely-confident,absolutely-confident


In [10]:
# Check the column structure and data distribution
print("Columns:", df_multipref_normal.columns.tolist())
print(f"\nDataset shape: {df_multipref_normal.shape}")
print(f"\nAnnotator distribution:")
print(df_multipref_normal['annotatorID'].value_counts().head(20))
print(f"\nPreference distribution (helpful):")
print(df_multipref_normal['helpful'].value_counts().sort_index())
print(f"\nPreference distribution (overall):")
print(df_multipref_normal['overall'].value_counts().sort_index())
print(f"\nConfidence distribution (helpful):")
print(df_multipref_normal['helpful_confidence'].value_counts())

Columns: ['comparison_id', 'prompt', 'response1', 'response2', 'annotatorID', 'helpful', 'truthful', 'harmless', 'overall', 'helpful_confidence', 'truthful_confidence', 'harmless_confidence', 'overall_confidence']

Dataset shape: (20922, 13)

Annotator distribution:
annotatorID
xenodochial_payne         531
stupefied_murdock         524
thirsty_northcutt         511
ecstatic_jones            460
affectionate_dubinsky     405
silly_khorana             403
cool_heyrovsky            401
zealous_boyd              401
heuristic_kirch           399
ecstatic_mcnulty          373
nervous_hypatia           367
hungry_clarke             366
clever_curie              351
peaceful_babbage          342
condescending_rosalind    341
gracious_kare             336
unruffled_hoover          331
nervous_nightingale       329
vigorous_lovelace         326
laughing_roentgen         322
Name: count, dtype: int64

Preference distribution (helpful):
helpful
-2    4946
-1    5114
 0    5586
 1    3272
 2    2

In [11]:
# Apply transformation for expert worker annotations
df_multipref_expert = transform_multipref_data(multipref, use_expert_annotations=True)

print(f"\nFirst few rows:")
df_multipref_expert.head(10)


First few rows:


,comparison_id,prompt,response1,response2,annotatorID,helpful,truthful,harmless,overall,helpful_confidence,truthful_confidence,harmless_confidence,overall_confidence
0,cdccd42baceb4358888fda8c23a7fb32,but it will provide acceleration in y directio...,To calculate the distance traveled in the y-di...,To find the distance traveled in the y-directi...,affectionate_shirley,2,0,0,2,absolutely-confident,absolutely-confident,absolutely-confident,fairly-confident
1,cdccd42baceb4358888fda8c23a7fb32,but it will provide acceleration in y directio...,To calculate the distance traveled in the y-di...,To find the distance traveled in the y-directi...,sad_jang,2,2,0,2,absolutely-confident,absolutely-confident,absolutely-confident,absolutely-confident
2,aa8af1e5ea024ad08e71273f5695993b,what are some fun sandwiches i can make for kids?,Here are a few fun sandwich ideas that kids mi...,"Kids love sandwiches, and making them fun can ...",cocky_davinci,-2,0,0,-2,absolutely-confident,absolutely-confident,absolutely-confident,absolutely-confident
3,aa8af1e5ea024ad08e71273f5695993b,what are some fun sandwiches i can make for kids?,Here are a few fun sandwich ideas that kids mi...,"Kids love sandwiches, and making them fun can ...",festive_mayer,0,0,0,0,absolutely-confident,absolutely-confident,absolutely-confident,absolutely-confident
4,6b9671fb535b447e898a24771d136306,what are some fun sandwiches i can make for kids?,Here are a few ideas for fun sandwiches that k...,Making sandwiches for kids is not only about p...,cocky_davinci,-2,0,0,-2,absolutely-confident,absolutely-confident,absolutely-confident,absolutely-confident
5,6b9671fb535b447e898a24771d136306,what are some fun sandwiches i can make for kids?,Here are a few ideas for fun sandwiches that k...,Making sandwiches for kids is not only about p...,upbeat_ptolemy,-2,0,0,-2,absolutely-confident,absolutely-confident,absolutely-confident,absolutely-confident
6,924dbf961d664b1c8634a0d831f0c8d4,what are august activiites for copenhagen rank...,"I'm sorry, but I don't have information about ...",Here are some popular activities to do in Co...,elastic_roentgen,-2,2,0,-1,fairly-confident,absolutely-confident,absolutely-confident,fairly-confident
7,924dbf961d664b1c8634a0d831f0c8d4,what are august activiites for copenhagen rank...,"I'm sorry, but I don't have information about ...",Here are some popular activities to do in Co...,peaceful_kilby,-1,0,0,-1,fairly-confident,fairly-confident,absolutely-confident,fairly-confident
8,fb0315fba0f1484994e2a9b3e7e1d4cb,what are august activiites for copenhagen rank...,Here are some popular activities to do in Cope...,"I'm sorry, but I don't have information about ...",festive_euclid,2,0,0,2,fairly-confident,absolutely-confident,absolutely-confident,fairly-confident
9,fb0315fba0f1484994e2a9b3e7e1d4cb,what are august activiites for copenhagen rank...,Here are some popular activities to do in Cope...,"I'm sorry, but I don't have information about ...",gifted_banach,2,-1,0,1,absolutely-confident,fairly-confident,absolutely-confident,absolutely-confident


In [12]:
df_multipref_normal['worker_type'] = 'normal'
df_multipref_expert['worker_type'] = 'expert'

# Merge
df_multipref_combined = pd.concat([df_multipref_normal, df_multipref_expert], ignore_index=True)
df_multipref_combined = df_multipref_combined.sort_values('comparison_id').reset_index(drop=True)

print(f"Total rows: {len(df_multipref_combined)}")
print(f"Normal workers: {len(df_multipref_normal)}")
print(f"Expert workers: {len(df_multipref_expert)}")

output_path_combined = "train/multipref_combined.json"
df_multipref_combined.to_json(output_path_combined, orient='records', indent=2)
print(f"\nCombined dataset saved to: {output_path_combined}")

Total rows: 41844
Normal workers: 20922
Expert workers: 20922

Combined dataset saved to: train/multipref_combined.json
